In [1]:
import sys
sys.path.insert(0, "/home/ibm-user/shared/safecircle")   # so "from src.explain import ..." works
sys.path.insert(0, "/home/ibm-user")                      # so Lab 2's Granite helper can be imported later

from src.explain import redact, explain

print(redact("Grandma it's me, call me at +1-555-0142"))
print(explain("test", {"risk": 0.9, "action": "hold_and_alert", "reasons": ["asks for gift cards"]}))

Grandma it's me, call me at [PHONE]
We paused this payment because the message asks for gift cards. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you already know, before sending any money.


In [7]:
import re
import pandas as pd

# Family words are the scam clue, not private data, so we never hide them
FAMILY_WORDS = {"Grandma", "Grandpa", "Grandmother", "Grandfather", "Nana", "Papa",
                "Mom", "Dad", "Mother", "Father", "Sir", "Madam", "Customer"}

# Names we know show up in our data. Add any others you spot in the test output.
KNOWN_NAMES = ["Rosa", "Walter", "Linda"]

CARD_RE  = re.compile(r"\b\d(?:[ -]?\d){11,18}\b")                    # 12-19 digits
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")             # name@site.com
LINK_RE  = re.compile(r"(?:https?://|www\.)\S+"
                      r"|\b[\w-]+(?:\.[\w-]+)*\.(?:com|net|org|xyz|info|io|co|us|ly)\b(?:/\S*)?")
PHONE_RE = re.compile(r"(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)?\b\d{3}[-.\s]\d{4}\b")
GREETING_RE = re.compile(r"\b(Hi|Hello|Hey|Dear|this is|This is|it's|It's|it is|It is)\s+([A-Z][a-z]+)")
TITLE_RE   = re.compile(r"\b((?:Dr|Mr|Mrs|Ms|Officer|Agent|Deputy|Sgt)\.?)\s+[A-Z][a-z]+")
ADDRESS_RE = re.compile(r"\b\d{1,5}\s+(?:[A-Z][a-z]+\s+){1,3}(?:St|Street|Ave|Avenue|Rd|Road|Blvd|Dr|Drive|Ln|Lane|Way|Ct|Court)\b")

def redact(text: str) -> str:
    """Hide card numbers, emails, links, phone numbers and names before text reaches the language model."""
    text = CARD_RE.sub("[CARD]", text)      # 1st: so the phone rule can't grab part of a card number
    text = EMAIL_RE.sub("[EMAIL]", text)    # 2nd: before links, so an email isn't split up
    text = LINK_RE.sub("[LINK]", text)
    text = PHONE_RE.sub("[PHONE]", text)
    text = ADDRESS_RE.sub("[ADDRESS]", text)
    text = TITLE_RE.sub(r"\1 [NAME]", text)          # "Dr. Patel" -> "Dr. [NAME]"

    def hide_name(match):
        if match.group(2) in FAMILY_WORDS:
            return match.group(0)                 # keep "Hi Grandma"
        return f"{match.group(1)} [NAME]"         # "It's Tyler" -> "It's [NAME]"
    text = GREETING_RE.sub(hide_name, text)

    for name in KNOWN_NAMES:
        text = re.sub(rf"\b{name}\b", "[NAME]", text)
    return text


print(redact("Appointment with Dr. Patel at 123 Main St. Call 555-0199. Hi Linda!"))

Appointment with Dr. [NAME] at [ADDRESS]. Call [PHONE]. Hi [NAME]!


In [8]:
extra = pd.read_csv("/home/ibm-user/shared/safecircle/data/extra_messages.csv")
scen  = pd.read_csv("/home/ibm-user/shared/safecircle/scenarios/messages.csv")
texts = pd.concat([extra["text"], scen["text"]], ignore_index=True)

results = pd.DataFrame({"before": texts, "after": texts.apply(redact)})
changed = results[results["before"] != results["after"]]
print(f"{len(results)} texts tested, {len(changed)} changed\n")

for _, row in changed.iterrows():
    print("BEFORE:", row["before"])
    print("AFTER: ", row["after"], "\n")

print("=== UNCHANGED (skim these for any missed name or number) ===")
for t in results.loc[results["before"] == results["after"], "before"]:
    print("-", t)

45 texts tested, 15 changed

BEFORE: This is Officer Daniels. Your grandson was arrested and needs $3,500 for bail. Pay with Apple gift cards and read me the numbers. This case is confidential.
AFTER:  This is [NAME] [NAME]. Your grandson was arrested and needs $3,500 for bail. Pay with Apple gift cards and read me the numbers. This case is confidential. 

BEFORE: URGENT: Your bank account has been locked due to suspicious activity. Verify your identity now at secure-acct-verify.xyz or your account will be closed.
AFTER:  URGENT: Your bank account has been locked due to suspicious activity. Verify your identity now at [LINK] or your account will be closed. 

BEFORE: Wells Fargo Alert: unusual sign-in detected. Confirm your card number and PIN at wf-secure-login.xyz within 24 hours to avoid suspension.
AFTER:  Wells Fargo Alert: unusual sign-in detected. Confirm your card number and PIN at [LINK] within 24 hours to avoid suspension. 

BEFORE: Your Zelle payment of $950 failed. To get a 

In [9]:
suspicious = results[results["after"].str.contains(r"\d{3}[-.\s]\d{4}|\d{7,}|@|https?://|\.xyz", regex=True)]
print(len(suspicious), "texts still look private")
suspicious["after"].tolist()

0 texts still look private


[]

In [2]:
from src.explain import redact
print(redact("Appointment with Dr. Patel at 123 Main St. Call 555-0199. Hi Linda!"))

Appointment with Dr. [NAME] at [ADDRESS]. Call [PHONE]. Hi [NAME]!


In [3]:
import time
from llama_cpp import Llama

GRANITE_PATH = "/home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf"

start = time.perf_counter()
llm = Llama(model_path=GRANITE_PATH, n_ctx=2048, n_threads=3, verbose=True)
print(f"\nLoaded in {time.perf_counter() - start:.1f} s")

llama_model_loader: loaded meta data with 43 key-value pairs and 402 tensors from /home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = granitehybrid
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Granite 4.0 H 350m Base
llama_model_loader: - kv   3:                           general.finetune str              = base
llama_model_loader: - kv   4:                           general.basename str              = granite-4.0-h
llama_model_loader: - kv   5:                         general.size_label str              = 350M
llama_model_loader: - kv   6:                            general.license str              = apache-2.0
llama_model_loader:


Loaded in 1.0 s


zDNN : NNPA = 1 | NNPA_PARMBLKFORMAT_0 = 1 | NNPA_PARMBLKFORMAT_1 = 1 | CPU : LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'tokenizer.ggml.add_bos_token': 'false', 'tokenizer.ggml.padding_token_id': '100256', 'tokenizer.ggml.bos_token_id': '100257', 'tokenizer.ggml.pre': 'dbrx', 'tokenizer.ggml.model': 'gpt2', 'granitehybrid.feed_forward_length': '2048', 'granitehybrid.residual_scale': '0.246000', 'granitehybrid.logit_scale': '3.000000', 'granitehybrid.block_count': '32', 'granitehybrid.attention.head_count': '12', 'tokenizer.ggml.unknown_token_id': '100269', 'granitehybrid.embedding_scale': '12.000000', 'general.size_label': '350M', 'granitehybrid.attention.scale': '0.015625', 'granitehybrid.embedding_length': '768', 'general.quantization_version': '2', 'general.basename': 'granite-4.0-h', 'granitehybrid.rope.freq_base': '10000.000000', 'general.license': 'apache-2.0', 'general.file_type': '1', 'granitehybrid.rope.scaling.finetuned': 'false', 'general.name': 'Granite 4.

In [4]:
def build_prompt(redacted_text, decision):
    reasons = "; ".join(decision["reasons"]) or "none"
    action = "PAUSED" if decision["action"] == "hold_and_alert" else "ALLOWED"
    return f"""Below are short, kind explanations for an older adult about why a payment was paused or allowed. Each uses simple words, 2 or 3 sentences, and never includes personal details.

Message: Hi Grandpa, it's [NAME]. I'm in jail and need $900 for bail. Please send it by Western Union and don't tell anyone.
Warning signs: pretends to be family; asks for a money transfer; asks you to keep it secret
Payment: PAUSED
Explanation: We paused this payment because the message pretends to be family, asks for money right away, and tells you to keep it secret. These are common signs of a scam. Please call your family member at a number you already know before sending anything.

Message: Your order has shipped and will arrive Thursday.
Warning signs: none
Payment: ALLOWED
Explanation: This payment looks normal for you, so it went through. No action is needed.

Message: {redacted_text}
Warning signs: {reasons}
Payment: {action}
Explanation:"""

In [5]:
from src.explain import redact

tests = {
    "rosa":   ("Grandma it's me, I'm in trouble. Please buy gift cards and text me the codes. Don't tell Mom.",
               {"action": "hold_and_alert", "reasons": ["asks for gift cards", "pretends to be family", "asks you to keep it secret", "new payee"]}),
    "walter": ("Your prescription is ready for pickup.",
               {"action": "allow", "reasons": []}),
    "linda":  ("Your bank account is locked. Verify your identity at the link below.",
               {"action": "hold_and_alert", "reasons": ["says your account is locked", "asks you to click a link", "wire to a new payee"]}),
}

for person, (text, decision) in tests.items():
    prompt = build_prompt(redact(text), decision)
    start = time.perf_counter()
    out = llm(prompt, max_tokens=80, temperature=0, stop=["\n\n", "\nMessage:"])
    seconds = time.perf_counter() - start
    print(f"--- {person} ({seconds:.1f} s) ---")
    print(out["choices"][0]["text"].strip(), "\n")

llama_perf_context_print:        load time =    5154.98 ms
llama_perf_context_print: prompt eval time =    5152.07 ms /   249 tokens (   20.69 ms per token,    48.33 tokens per second)
llama_perf_context_print:        eval time =    1514.37 ms /    16 runs   (   94.65 ms per token,    10.57 tokens per second)
llama_perf_context_print:       total time =    6679.22 ms /   265 tokens
llama_perf_context_print:    graphs reused =         14
Llama.generate: 192 prefix-match found but partial kv removal not supported, re-evaluating full prompt


--- rosa (6.7 s) ---
This payment looks normal for you, so it went through. No action is needed. 



llama_perf_context_print:        load time =    5154.98 ms
llama_perf_context_print: prompt eval time =    4133.76 ms /   211 tokens (   19.59 ms per token,    51.04 tokens per second)
llama_perf_context_print:        eval time =    1521.86 ms /    16 runs   (   95.12 ms per token,    10.51 tokens per second)
llama_perf_context_print:       total time =    5667.60 ms /   227 tokens
llama_perf_context_print:    graphs reused =         15
Llama.generate: 193 prefix-match found but partial kv removal not supported, re-evaluating full prompt


--- walter (5.7 s) ---
This payment looks normal for you, so it went through. No action is needed. 



llama_perf_context_print:        load time =    5154.98 ms
llama_perf_context_print: prompt eval time =    5237.39 ms /   236 tokens (   22.19 ms per token,    45.06 tokens per second)
llama_perf_context_print:        eval time =    1472.67 ms /    16 runs   (   92.04 ms per token,    10.86 tokens per second)
llama_perf_context_print:       total time =    6722.88 ms /   252 tokens
llama_perf_context_print:    graphs reused =         15


--- linda (6.7 s) ---
This payment looks normal for you, so it went through. No action is needed. 



In [6]:
PAUSED_EXAMPLES = """Below are short, kind explanations for an older adult about why a payment was paused. Each uses simple words, 2 or 3 sentences, and never includes personal details.

Message: Hi Grandpa, it's [NAME]. I'm in jail and need $900 for bail. Please send it by Western Union and don't tell anyone.
Warning signs: claims to be a family member; asks for a money transfer; asks you to keep it secret
Explanation: We paused this payment because the message claims to be a family member, asks for money right away, and tells you to keep it secret. These are common signs of a scam. Please call your family member at a number you already know before sending anything.

Message: Your online store account is suspended. Confirm your card at [LINK] within 1 hour.
Warning signs: says your account has a problem; asks you to click a link; uses urgent language
Explanation: We paused this payment because the message says your account has a problem, pushes you to click a link, and rushes you. Real companies don't ask for card details this way. Please call the company using the number on your card or statement.
"""

def build_paused_prompt(redacted_text, decision):
    reasons = "; ".join(decision["reasons"])
    return PAUSED_EXAMPLES + f"\nMessage: {redacted_text}\nWarning signs: {reasons}\nExplanation:"

# Akshi's real reasons + the kind of reasons the Scorer will add
paused_tests = {
    "rosa":  ("Grandma it's me, I'm in trouble. Please buy gift cards and text me the codes. Don't tell Mom.",
              {"action": "hold_and_alert", "reasons": ["asks you to text a code", "claims to be a family member", "gift-card merchant", "new payee"]}),
    "linda": ("Your bank account is locked. Verify your identity at the link below.",
              {"action": "hold_and_alert", "reasons": ["uses urgent language", "mentions your account", "wire transfer", "new payee"]}),
}

for person, (text, decision) in paused_tests.items():
    start = time.perf_counter()
    out = llm(build_paused_prompt(redact(text), decision), max_tokens=90, temperature=0, stop=["\n\n", "\nMessage:"])
    print(f"--- {person} ({time.perf_counter() - start:.1f} s) ---")
    print(out["choices"][0]["text"].strip(), "\n")

Llama.generate: 16 prefix-match found but partial kv removal not supported, re-evaluating full prompt
llama_perf_context_print:        load time =    5154.98 ms
llama_perf_context_print: prompt eval time =    5969.12 ms /   292 tokens (   20.44 ms per token,    48.92 tokens per second)
llama_perf_context_print:        eval time =    4455.51 ms /    49 runs   (   90.93 ms per token,    11.00 tokens per second)
llama_perf_context_print:       total time =   10457.08 ms /   341 tokens
llama_perf_context_print:    graphs reused =         48
Llama.generate: 240 prefix-match found but partial kv removal not supported, re-evaluating full prompt


--- rosa (10.5 s) ---
We paused this payment because the message claims to be a family member, asks you to text a code, and uses urgent language. Real companies don't ask for card details this way. Please call the company using the number on your card or statement. 



llama_perf_context_print:        load time =    5154.98 ms
llama_perf_context_print: prompt eval time =    5570.94 ms /   274 tokens (   20.33 ms per token,    49.18 tokens per second)
llama_perf_context_print:        eval time =    4216.98 ms /    47 runs   (   89.72 ms per token,    11.15 tokens per second)
llama_perf_context_print:       total time =    9816.83 ms /   321 tokens
llama_perf_context_print:    graphs reused =         46


--- linda (9.8 s) ---
We paused this payment because the message uses urgent language, mentions your account, and asks you to verify your identity. Real companies don't ask for your account information this way. Please call your bank using the number on your card or statement. 



In [7]:
# Message reasons start with a verb ("asks you to...", "claims to be...");
# payment reasons are short labels ("new payee", "gift-card merchant").
VERB_STARTS = ("asks", "claims", "says", "uses", "mentions", "pretends", "tells", "threatens", "offers", "promises")

def _join(items):
    items = list(items)
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]

def template_explain(decision):
    """Plain-English explanation built only from the decision. No AI model, so it never makes things up."""
    if decision["action"] != "hold_and_alert":
        return "This payment looks normal for you, so it went through. No action is needed."

    reasons = [r for r in decision.get("reasons", []) if not r.startswith("uses the word")]
    msg_reasons = [r for r in reasons if r.startswith(VERB_STARTS)][:3]
    pay_reasons = [r for r in reasons if not r.startswith(VERB_STARTS)][:3]

    parts = []
    if msg_reasons:
        parts.append(f"We paused this payment because the message {_join(msg_reasons)}.")
    else:
        parts.append("We paused this payment because it doesn't match your usual spending.")
    if pay_reasons:
        parts.append(f"The payment itself also looked unusual: {_join(pay_reasons)}.")
    parts.append("These are common signs of a scam, so we let your trusted contact know. "
                 "Please call them, or call the person at a number you already know, before sending any money.")
    return " ".join(parts)

for person, (text, decision) in {**paused_tests, "walter": tests["walter"]}.items():
    print(f"--- {person} ---\n{template_explain(decision)}\n")

--- rosa ---
We paused this payment because the message asks you to text a code and claims to be a family member. The payment itself also looked unusual: gift-card merchant and new payee. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you already know, before sending any money.

--- linda ---
We paused this payment because the message uses urgent language and mentions your account. The payment itself also looked unusual: wire transfer and new payee. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you already know, before sending any money.

--- walter ---
This payment looks normal for you, so it went through. No action is needed.



In [2]:
import time
from src.explain import redact, explain

scenarios = {
    "rosa":   ("Grandma it's me, I'm in trouble. Please buy gift cards and text me the codes. Don't tell Mom.",
               {"action": "hold_and_alert", "reasons": ["asks you to text a code", "claims to be a family member", "gift-card merchant", "new payee"]}),
    "walter": ("Your prescription is ready for pickup.",
               {"action": "allow", "reasons": []}),
    "linda":  ("Your bank account is locked. Verify your identity at the link below.",
               {"action": "hold_and_alert", "reasons": ["uses urgent language", "mentions your account", "wire transfer", "new payee"]}),
}

print("=== TEMPLATE (default) ===")
for person, (text, decision) in scenarios.items():
    print(f"{person}: {explain(redact(text), decision)}\n")

print("=== GRANITE (optional) ===")
for person in ["rosa", "linda"]:
    text, decision = scenarios[person]
    start = time.perf_counter()
    result = explain(redact(text), decision, use_granite=True)
    print(f"{person} ({time.perf_counter() - start:.1f} s): {result}\n")

=== TEMPLATE (default) ===
rosa: We paused this payment because the message asks you to text a code and claims to be a family member. The payment itself also looked unusual: gift-card merchant and new payee. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you already know, before sending any money.

walter: This payment looks normal for you, so it went through. No action is needed.

linda: We paused this payment because the message uses urgent language and mentions your account. The payment itself also looked unusual: wire transfer and new payee. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you already know, before sending any money.

=== GRANITE (optional) ===


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


rosa (11.2 s): We paused this payment because the message claims to be a family member, asks you to text a code, and uses urgent language. Real companies don't ask for card details this way. Please call the company using the number on your card or statement.

linda (9.9 s): We paused this payment because the message uses urgent language, mentions your account, and asks you to verify your identity. Real companies don't ask for your account information this way. Please call your bank using the number on your card or statement.

